# Estudio 06 · Diagnóstico frente a la persistencia (solo lectura)

Pedido de Rosa del 2026-11-08, después de elegir L + 5a + d3 como base de la V3 (b3 retadora). **No se entrena ni se
preregistra nada**: se reevalúan con el protocolo del estudio (Platt por horizonte con los bloques anteriores y la mezcla
5a con la w de los bloques anteriores) las predicciones guardadas de L + 5a y b3 (`temporada_2026`) y de d3
(`matriz_clima`), 5 semillas; las cifras son medianas entre semillas.

1. Error del conteo por fase de la curva regional (valle, subida, pico, bajada) en `temporada_2026` y `posterior_2026`.
2. Peso w de la mezcla 5a por bloque y la razón MAE con w = 0, w = 1 y la w retrospectiva (lectura).
3. Probabilidad calibrada antes de los inicios limpios de 2026 frente a la señal de la persistencia.
4. Calibración en 2026: probabilidad media frente a la tasa observada de brotes.

Las fases salen de `denguekay.evaluation.episodes.curve_phase` (curva regional del bloque: valle < 10 % del máximo,
pico >= 60 %, subida antes de la semana del máximo y bajada después). `posterior_2026` cubre pocas semanas
(2026-08-30 a 2026-10-11), así que sus fases son cortas.

In [1]:
import json
import sys

import numpy as np
import pandas as pd

from denguekay.config.paths import MODELS, RESEARCH

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import diagnosis_persistence as dp
import phase5_counts as p5
import phase_2026 as p26
import preregistration as pr

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
METRICS = STUDY / "docs" / "metricas" / "diagnostico_persistencia.json"
EXP = MODELS / "experimentos" / "mejora_modelo"
PATHS = {"L + 5a": EXP / "temporada_2026" / "predicciones_L_5a.csv.gz",
         "b3": EXP / "temporada_2026" / "predicciones_b3.csv.gz",
         "d3": EXP / "matriz_clima" / "predicciones_d3.csv.gz"}
inp = p26.inputs_2026()
series = inp["serie"]

runs = {}
for name, path in PATHS.items():
    raw = p26.read_saved(path)
    runs[name] = {}
    for seed, pred in raw.items():
        ev = p5.evaluate_mixture(pred, series)
        runs[name][seed] = {"datos": dp.with_counts(ev["datos"], pred), "w": ev["w"], "antelacion": ev["antelacion"]}
{k: sorted(v) for k, v in runs.items()}

{'L + 5a': [17, 29, 41, 53, 67],
 'b3': [17, 29, 41, 53, 67],
 'd3': [17, 29, 41, 53, 67]}

## 1 · Error del conteo por fase de la curva

`sesgo` = media de (predicho − observado): negativo = se queda corto. `suma_*_sobre_real` = casos predichos / casos observados en la fase. `mezcla` es el conteo publicado (5a), `modelo` la μ sola (w = 1).

In [2]:
keys = ["horizonte", "bloque", "fase"]
phase = pd.concat([dp.seed_median_frame([dp.phase_errors(r["datos"]) for r in runs[k].values()], keys).assign(variante=k)
                   for k in PATHS], ignore_index=True)
cols = ["variante", "horizonte", "bloque", "fase", "semanas", "casos_observados", "sesgo_mezcla", "sesgo_modelo",
        "sesgo_persistencia", "suma_mezcla_sobre_real", "suma_modelo_sobre_real", "razon_mae_mezcla", "razon_mae_modelo"]
phase[cols].sort_values(["bloque", "horizonte", "fase", "variante"]).round(3)

,variante,horizonte,bloque,fase,semanas,casos_observados,sesgo_mezcla,sesgo_modelo,sesgo_persistencia,suma_mezcla_sobre_real,suma_modelo_sobre_real,razon_mae_mezcla,razon_mae_modelo
0,L + 5a,2,posterior_2026,pico,3.0,3337.0,-7.589,-8.531,-6.118,0.557,0.501,1.146,1.254
15,b3,2,posterior_2026,pico,3.0,3337.0,-7.280,-7.485,-6.118,0.575,0.563,1.119,1.144
30,d3,2,posterior_2026,pico,3.0,3337.0,-6.026,-5.991,-6.118,0.648,0.650,0.993,0.990
1,L + 5a,2,posterior_2026,subida,4.0,2294.0,-2.343,-2.983,-1.342,0.734,0.662,1.081,1.182
16,b3,2,posterior_2026,subida,4.0,2294.0,-2.708,-2.968,-1.342,0.693,0.664,1.136,1.184
31,d3,2,posterior_2026,subida,4.0,2294.0,-1.828,-2.016,-1.342,0.793,0.771,1.047,1.070
5,L + 5a,3,posterior_2026,pico,3.0,3337.0,-9.581,-10.453,-8.031,0.440,0.389,1.141,1.221
20,b3,3,posterior_2026,pico,3.0,3337.0,-9.717,-9.992,-8.031,0.432,0.416,1.160,1.186
35,d3,3,posterior_2026,pico,3.0,3337.0,-7.875,-7.815,-8.031,0.540,0.543,0.984,0.979
6,L + 5a,3,posterior_2026,subida,4.0,2294.0,-3.049,-3.806,-1.712,0.654,0.569,1.132,1.268


## 2 · Peso w de la mezcla 5a y razón MAE con w = 0 / w = 1 (lectura)

w por bloque con las filas OOS de los bloques anteriores (lo que usa el protocolo). La w retrospectiva es la de la grilla con menor MAE en el propio bloque: usa el futuro y es solo una lectura.

In [3]:
w_all = pd.concat([pd.concat([r["w"].assign(semilla=s) for s, r in runs[k].items()]).assign(variante=k) for k in PATHS])
w_table = w_all.groupby(["variante", "horizonte", "bloque"]).w.median().unstack("bloque")
order = [c for c in w_table.columns if c.startswith("temporada_")] + [c for c in w_table.columns if not c.startswith("temporada_")]
w_table = w_table[sorted(order, key=lambda b: (b == "posterior_2026", b))]
w_table.round(2)

bloque              calendario_2025  temporada_2008  temporada_2009  temporada_2010  temporada_2011  temporada_2012  temporada_2013  temporada_2014  temporada_2015  temporada_2016  temporada_2017  temporada_2018  temporada_2019  temporada_2020  \
variante horizonte                                                                                                                                                                                                                                    
L + 5a   2                     0.63             1.0            1.00            0.88            0.66            0.62            0.62            0.62            0.59            0.61            0.58            0.83            0.79            0.78   
         3                     0.66             1.0            1.00            1.00            0.74            0.68            0.66            0.70            0.65            0.65            0.59            0.84            0.82            0.81   
         4                     0.71             1.0            0.97            1.00            0.75            0.73            0.72            0.73            0.69            0.69            0.66            0.87            0.85            0.85   
b3       2                     0.87             1.0            1.00            0.91            0.76            0.69            0.69            0.70            0.69            0.69            0.64            0.93            0.91            0.90   
         3                     0.88             1.0            1.00            1.00            0.81            0.78            0.78            0.79            0.71            0.69            0.67            0.93            0.91            0.89   
         4                     0.92             1.0            1.00            1.00            0.80            0.78            0.78            0.79            0.78            0.78            0.73            0.96            0.95            0.94   
d3       2                     0.73             1.0            0.87            0.66            0.69            0.62            0.61            0.62            0.60            0.61            0.56            0.80            0.77            0.75   
         3                     0.73             1.0            1.00            0.93            0.79            0.74            0.72            0.75            0.70            0.64            0.55            0.86            0.84            0.83   
         4                     0.76             1.0            0.99            1.00            0.74            0.73            0.73            0.74            0.72            0.71            0.64            0.87            0.86            0.85   

bloque              temporada_2021  temporada_2022  temporada_2023  temporada_2024  temporada_2026  posterior_2026  
variante horizonte                                                                                                  
L + 5a   2                    0.75            0.74            0.73            0.73            0.62            0.60  
         3                    0.80            0.80            0.79            0.77            0.65            0.63  
         4                    0.84            0.84            0.81            0.85            0.68            0.66  
b3       2                    0.86            0.84            0.84            0.86            0.86            0.84  
         3                    0.88            0.86            0.87            0.88            0.88            0.87  
         4                    0.93            0.90            0.91            0.92            0.91            0.90  
d3       2                    0.73            0.73            0.73            0.83            0.72            0.72  
         3                    0.83            0.82            0.80            0.84            0.73            0.72  
         4                    0.84            0.84            0.81            0.90            0.75            0.

In [4]:
mix = pd.concat([dp.seed_median_frame([dp.mixture_reading(r["datos"], r["w"]) for r in runs[k].values()],
                                       ["horizonte", "bloque"]).assign(variante=k) for k in PATHS], ignore_index=True)
mix[["variante", "horizonte", "bloque", "w_usado", "razon_mae_w_usado", "razon_mae_w1", "razon_mae_w0",
     "w_retrospectivo", "razon_mae_w_retrospectivo"]].sort_values(["bloque", "horizonte", "variante"]).round(3)

,variante,horizonte,bloque,w_usado,razon_mae_w_usado,razon_mae_w1,razon_mae_w0,w_retrospectivo,razon_mae_w_retrospectivo
0,L + 5a,2,posterior_2026,0.60,1.122,1.228,1.0,0.00,1.000
6,b3,2,posterior_2026,0.84,1.126,1.158,1.0,0.00,1.000
12,d3,2,posterior_2026,0.72,1.012,1.019,1.0,0.00,1.000
2,L + 5a,3,posterior_2026,0.63,1.138,1.238,1.0,0.00,1.000
8,b3,3,posterior_2026,0.87,1.180,1.217,1.0,0.00,1.000
14,d3,3,posterior_2026,0.72,1.014,1.028,1.0,0.00,1.000
4,L + 5a,4,posterior_2026,0.66,1.191,1.305,1.0,0.00,1.000
10,b3,4,posterior_2026,0.90,1.284,1.317,1.0,0.00,1.000
16,d3,4,posterior_2026,0.74,1.068,1.104,1.0,0.00,1.000
1,L + 5a,2,temporada_2026,0.62,1.064,1.207,1.0,0.11,0.996


## 3 · Probabilidad antes de los inicios limpios de 2026

Inicios limpios del protocolo (sin brote en las 8 semanas previas del distrito). `p_semana_inicio`: probabilidad calibrada emitida h semanas antes para la semana del inicio. `p_max_previas`: la mayor para las 4 semanas objetivo anteriores al inicio. Antelación: semanas entre la primera alerta y el inicio (>= 2 = avisado). La persistencia es la misma para las tres variantes.

In [5]:
okeys = ["horizonte", "ubigeo", "inicio"]
def onsets_seed(r):
    t = dp.onset_probabilities(r["datos"], r["antelacion"])
    return t.assign(alerta_persistencia_semana_inicio=t.alerta_persistencia_semana_inicio.astype(int))
onset = {k: dp.seed_median_frame([onsets_seed(r) for r in runs[k].values()], okeys) for k in PATHS}
summary = pd.concat([dp.onset_summary(onset[k].assign(alerta_persistencia_semana_inicio=onset[k].alerta_persistencia_semana_inicio.ge(0.5)))
                     .assign(variante=k) for k in PATHS], ignore_index=True)
summary.round(3)

,horizonte,inicios,p_mediana_semana_inicio,p_mediana_max_previas,con_p_050_semana_inicio,persistencia_alerta_semana_inicio,avisados_050,avisados_persistencia,variante
0,2,20,0.180,0.113,2,4,2,4,L + 5a
1,3,20,0.249,0.060,1,8,2,9,L + 5a
2,4,20,0.071,0.032,0,4,2,8,L + 5a
3,2,20,0.198,0.093,2,4,2,4,b3
4,3,20,0.267,0.073,2,8,3,9,b3
5,4,20,0.091,0.029,0,4,2,8,b3
6,2,20,0.210,0.119,3,4,3,4,d3
7,3,20,0.286,0.100,3,8,4,9,d3
8,4,20,0.085,0.032,1,4,5,8,d3


In [6]:
wide = onset["L + 5a"][okeys + ["persistencia_semana_inicio", "umbral", "antelacion_persistencia", "p_semana_inicio"]].rename(
    columns={"p_semana_inicio": "p_L_5a"})
for k in ("b3", "d3"):
    wide = wide.merge(onset[k][okeys + ["p_semana_inicio"]].rename(columns={"p_semana_inicio": f"p_{k}"}), on=okeys)
for k in PATHS:
    wide = wide.merge(onset[k][okeys + ["antelacion_050"]].rename(columns={"antelacion_050": f"antel_{k.replace(' + ', '_')}"}), on=okeys)
wide.sort_values(["horizonte", "inicio", "ubigeo"]).round(3)

C:\Users\Rosa\AppData\Local\Temp\ipykernel_12100\632763979.py:7: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  wide.sort_values(["horizonte", "inicio", "ubigeo"]).round(3)


,horizonte,ubigeo,inicio,persistencia_semana_inicio,umbral,antelacion_persistencia,p_L_5a,p_b3,p_d3,antel_L_5a,antel_b3,antel_d3
13,2,200601,2025-11-09,3.0,4.686,NaN,0.274,0.324,0.273,NaN,NaN,NaN
10,2,200405,2026-03-22,16.0,30.146,-3.0,0.353,0.338,0.295,-2.0,-2.0,-2.0
9,2,200404,2026-06-21,15.0,15.413,1.0,0.301,0.329,0.301,1.0,1.0,1.0
2,2,200111,2026-06-28,7.0,3.156,2.0,0.668,0.699,0.706,2.0,2.0,2.0
8,2,200402,2026-07-05,0.0,0.871,1.0,0.073,0.089,0.071,0.0,0.0,0.0
14,2,200601,2026-07-05,38.0,50.963,1.0,0.158,0.190,0.287,0.0,0.0,1.0
16,2,200604,2026-07-05,1.0,2.612,0.0,0.155,0.196,0.193,0.0,0.0,0.0
5,2,200207,2026-07-12,0.0,5.225,0.0,0.000,0.000,0.000,0.0,0.0,0.0
6,2,200210,2026-07-12,2.0,5.512,1.0,0.059,0.056,0.060,1.0,1.0,1.0
7,2,200401,2026-07-12,75.0,70.841,2.0,0.303,0.330,0.366,0.0,1.0,1.0


## 4 · Calibración en 2026

Probabilidad calibrada media frente a la tasa observada de semanas brote (razón > 1 = sobrepronostica).

In [7]:
cal = pd.concat([dp.seed_median_frame([dp.calibration_in_large(r["datos"]) for r in runs[k].values()],
                                       ["horizonte", "bloque"]).assign(variante=k) for k in PATHS], ignore_index=True)
cal.sort_values(["bloque", "horizonte", "variante"]).round(4)

,horizonte,bloque,n,brotes,tasa_observada,prob_media,razon_prob_tasa,prob_media_en_brote,prob_media_sin_brote,variante
0,2,posterior_2026,455.0,167.0,0.3670,0.2733,0.7446,0.6313,0.0656,L + 5a
6,2,posterior_2026,455.0,167.0,0.3670,0.2866,0.7809,0.6557,0.0723,b3
12,2,posterior_2026,455.0,167.0,0.3670,0.2967,0.8085,0.6730,0.0786,d3
2,3,posterior_2026,455.0,167.0,0.3670,0.2416,0.6583,0.5552,0.0598,L + 5a
8,3,posterior_2026,455.0,167.0,0.3670,0.2501,0.6813,0.5718,0.0635,b3
14,3,posterior_2026,455.0,167.0,0.3670,0.2667,0.7267,0.6024,0.0720,d3
4,4,posterior_2026,455.0,167.0,0.3670,0.2114,0.5761,0.4913,0.0492,L + 5a
10,4,posterior_2026,455.0,167.0,0.3670,0.2167,0.5903,0.5022,0.0511,b3
16,4,posterior_2026,455.0,167.0,0.3670,0.2355,0.6417,0.5400,0.0590,d3
1,2,temporada_2026,3445.0,89.0,0.0258,0.0241,0.9326,0.5241,0.0108,L + 5a


## 5 · Cifras

In [8]:
def records(df):
    return json.loads(df.to_json(orient="records", date_format="iso", force_ascii=False))

out = {
    "descripcion": "Diagnóstico de solo lectura frente a la persistencia (L + 5a, b3, d3); medianas de 5 semillas.",
    "predicciones": {k: str(v.relative_to(MODELS.parent)) for k, v in PATHS.items()},
    "errores_por_fase": records(phase),
    "w_por_bloque": records(w_table.reset_index()),
    "lectura_mezcla": records(mix),
    "inicios_resumen": records(summary),
    "inicios_detalle": records(wide),
    "calibracion": records(cal),
}
METRICS.write_text(json.dumps(out, ensure_ascii=False, indent=1), encoding="utf-8")
print(METRICS.name, len(out["inicios_detalle"]))

diagnostico_persistencia.json 60
